In [1]:
import numpy as np
import torch

In [2]:
X = np.array([160, 170, 180, 190])
y = np.array([0, 0, 1, 1])

X, y

(array([160, 170, 180, 190]), array([0, 0, 1, 1]))

In [3]:
X_mean = np.mean(X)
X_std = np.std(X)

X_norm = (X - X_mean) / X_std
X_mean, X_std, X_norm

(np.float64(175.0),
 np.float64(11.180339887498949),
 array([-1.34164079, -0.4472136 ,  0.4472136 ,  1.34164079]))

In [4]:
X_norm_tensor = torch.tensor(X_norm, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

X_norm_tensor = X_norm_tensor.reshape(-1, 1)
y_tensor = y_tensor.reshape(-1, 1)

X_norm_tensor, y_tensor

(tensor([[-1.3416],
         [-0.4472],
         [ 0.4472],
         [ 1.3416]]),
 tensor([[0.],
         [0.],
         [1.],
         [1.]]))

In [5]:
X_norm_tensor.shape, y_tensor.shape

(torch.Size([4, 1]), torch.Size([4, 1]))

In [6]:
# model 정의
class PerceptronModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # 선형. 입력1 출력1
        self.linear = torch.nn.Linear(1, 1)
    
    # 이전의 시그모이드 과정 삭제
    def forward(self, x):
        H = self.linear(x)
        return H
    


In [7]:
# 랜덤 초기값 고정
torch.manual_seed(42)
    
model = PerceptronModel()
model.linear.weight, model.linear.bias, model

(Parameter containing:
 tensor([[0.7645]], requires_grad=True),
 Parameter containing:
 tensor([0.8300], requires_grad=True),
 PerceptronModel(
   (linear): Linear(in_features=1, out_features=1, bias=True)
 ))

In [8]:
# 학습전 출력 확인
with torch.no_grad():
    H_test = model(X_norm_tensor)
    z_test = torch.sigmoid(H_test)

H_test.shape, H_test, z_test.shape, z_test

(torch.Size([4, 1]),
 tensor([[-0.1957],
         [ 0.4881],
         [ 1.1719],
         [ 1.8557]]),
 torch.Size([4, 1]),
 tensor([[0.4512],
         [0.6197],
         [0.7635],
         [0.8648]]))

In [9]:
# 학습 설정
learning_rate = 0.1
epochs = 1000
criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
print(list(model.parameters()))

[Parameter containing:
tensor([[0.7645]], requires_grad=True), Parameter containing:
tensor([0.8300], requires_grad=True)]


In [10]:
for epoch in range(epochs):
    # 0초기화
    optimizer.zero_grad()
    # 선형 계산
    H = model(X_norm_tensor)
    # bcdwithlogitsloss 계산(z가 아니라 H 들어감)
    mean_cost = criterion(H, y_tensor)
    # grad 계산
    mean_cost.backward()
    # 파라미터 업데이트
    optimizer.step()

    if epoch % 100 == 0 or epoch == epochs - 1:
        print(
            f'epoch={epoch}, '
            f'Cost={mean_cost.item():.6f}'
            f'weight(a)={model.linear.weight.item():.6f}'
            f'bias(b)={model.linear.bias.item():.6f}'
        )
    
    if epoch < 3:
        print(
            f'  (확인용) model.linear.weight.grad={model.linear.weight.grad.item():.6f}'
            f'model.linear.bias.grad={model.linear.bias.grad.item():.6f}'
        )

epoch=0, Cost=0.495464weight(a)=0.793780bias(b)=0.812529
  (확인용) model.linear.weight.grad=-0.292415model.linear.bias.grad=0.174793
  (확인용) model.linear.weight.grad=-0.286153model.linear.bias.grad=0.169918
  (확인용) model.linear.weight.grad=-0.280072model.linear.bias.grad=0.165171
epoch=100, Cost=0.178670weight(a)=2.290082bias(b)=0.173212
epoch=200, Cost=0.125357weight(a)=3.002210bias(b)=0.061586
epoch=300, Cost=0.099283weight(a)=3.509002bias(b)=0.026837
epoch=400, Cost=0.082901weight(a)=3.912263bias(b)=0.013229
epoch=500, Cost=0.071398weight(a)=4.250606bias(b)=0.007116
epoch=600, Cost=0.062789weight(a)=4.543496bias(b)=0.004091
epoch=700, Cost=0.056068weight(a)=4.802371bias(b)=0.002480
epoch=800, Cost=0.050660weight(a)=5.034644bias(b)=0.001570
epoch=900, Cost=0.046207weight(a)=5.245449bias(b)=0.001031
epoch=999, Cost=0.042507weight(a)=5.436657bias(b)=0.000701


In [11]:
print('model.linear.weight:', model.linear.weight)
print('model.linear.bias:', model.linear.bias)

print('model.linear.weight.grad:', model.linear.weight.grad)
print('model.linear.bias.grad:', model.linear.bias.grad)

model.linear.weight: Parameter containing:
tensor([[5.4367]], requires_grad=True)
model.linear.bias: Parameter containing:
tensor([0.0007], requires_grad=True)
model.linear.weight.grad: tensor([[-0.0185]])
model.linear.bias.grad: tensor([2.6390e-05])


In [ ]:
input_height = 185
input_norm = (input_height - X_mean) / X_std

with torch.no_grad():
    input_norm_tensor = torch.tensor([[input_norm]], dtype=torch.float32)
    print('input_norm_tensor shape:', input_norm_tensor.shape)

    H_new = model(input_norm_tensor)

    probability = torch.sigmoid(H_new)

    pred = 1 if probability.item() >= 0.5 else 0

print('H_new:', H_new.item())
print('probability:', probability.item())
print('prediction:', pred)
print(f'\n키가 {input_height}cm인 사람의 농구선수 확률(probability): {probability.item():.4f}')

input_norm_tensor shape: torch.Size([1, 1])
H_new: 4.863395690917969
probability: 0.9923350214958191
prediction: 1

키가 185cm인 사람의 농구선수 확률(probability): 0.9923
